## Reading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(color_codes = True,
            style='darkgrid')

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/uom190346a/sleep-health-and-lifestyle-dataset/Sleep_health_and_lifestyle_dataset.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()  #for type of values and null values

###Observation
 - 5 columns are object datatype
 - sum null values in Sleep Disorder(target Variable)

In [ ]:
df['Sleep Disorder'].unique()

In [ ]:
# replace nan with no disorder
df['Sleep Disorder']=df['Sleep Disorder'].where(pd.notnull(df['Sleep Disorder']), 'No Disorder')

# pd.notnull(df['Sleep Disorder'])

In [ ]:
df.info()

In [ ]:
df.describe()  #statistical info for numerical variables

In [ ]:
df.describe(include='O')   #statistical info for object variale

# EDA

In [ ]:
col_name = df.columns
col_name

In [ ]:
# unique values
df.nunique()

In [ ]:
# droping person ID
df.drop(columns='Person ID', inplace = True)

In [ ]:
df.columns

In [ ]:
df[['Upper BP', 'Lower BP']] = df['Blood Pressure'].str.split('/', expand = True).astype(int)
df.drop(columns = 'Blood Pressure', inplace = True)
df.head()

In [ ]:
# PAIR Plot
sns.pairplot(data = df, hue='Sleep Disorder', palette = 'dark')

In [ ]:
# histogram for people distribution
fig = px.histogram(df, x = 'Sleep Disorder',
             barmode='group', color = 'Sleep Disorder', text_auto=True)

fig.update_layout(title = 'Distribution of people based on sleep disorder',
                  title_font = {'size':30},
                  showlegend = True)
fig.update_yaxes(showgrid = False)
fig.show()

In [ ]:
# gender
df.groupby('Sleep Disorder')['Gender'].value_counts()

In [ ]:

sns.countplot(x = 'Gender', hue='Sleep Disorder', data = df)


### Observation
1. Normal Males are more than women
2. Sleep APnea is more in females than in males
3. Insomania is more in males than in females

In [ ]:
# gender
df.groupby('Sleep Disorder')['Gender'].value_counts().plot.pie(autopct = '%1.1f%%',
                                                               figsize = (15,6),
                                                               colors = ['#ed72be', '#ed72e7',
                                                                         '#a678b0', '#b567c7','#a62357'])

plt.title('Sleep Disorder and Sex Relationship')
plt.axis('equal')
plt.show()

In [ ]:
df.columns

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(exclude='object').columns

In [ ]:
num_cols, cat_cols

In [ ]:
# plot categoriacal variables
fig,axs = plt.subplots(nrows=1, ncols = 3, figsize=(15,6))
axs = axs.flatten()

#barplot
for i, var in enumerate(cat_cols[:3]):
  sns.countplot(x = var, hue='Sleep Disorder', data = df, ax = axs[i])
  axs[i].set_xticklabels(axs[i].get_xticklabels(), rotation = 90)


fig.tight_layout()
plt.show()

### Observation
1. Doctors have normal sleeps
2. Nurses are suffereing from Sleep Apnea.
3. Salesperson has highest Insomania
4. Overwieght ppl has higher chances of sleep disorder

In [ ]:
sns.histplot(x = df['Occupation'], hue = 'Sleep Disorder', data = df, multiple='fill',
             kde = False, element = 'bars',fill=True)

In [ ]:
# plot categoriacal variables
fig,axs = plt.subplots(nrows=1, ncols = 3, figsize=(15,6))
axs = axs.flatten()

#barplot
for i, var in enumerate(cat_cols[:3]):
  sns.histplot(x = var, hue = 'Sleep Disorder', data = df,ax= axs[i], multiple='fill',
             kde = False, element = 'bars',fill=True)
  axs[i].set_xticklabels(axs[i].get_xticklabels(), rotation = 90)
  axs[i].set_xlabel(var)


fig.tight_layout()
plt.show()

In [ ]:
num_cols

In [ ]:
fig,axs = plt.subplots(nrows=3, ncols = 3, figsize=(20,10))
axs = axs.flatten()


for i, var in enumerate(num_cols):
  sns.boxplot(x = var, data = df, ax=axs[i])

fig.tight_layout()
fig.show()

In [ ]:
fig,axs = plt.subplots(nrows=3, ncols = 3, figsize=(20,20))
axs = axs.flatten()


for i, var in enumerate(num_cols):
  sns.boxplot(y = var, x= 'Sleep Disorder', hue='Sleep Disorder',data = df, ax=axs[i])

fig.tight_layout()
fig.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(x = 'Stress Level', hue='Sleep Disorder', data = df)


In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(x = 'Quality of Sleep', hue='Sleep Disorder', data = df)


In [ ]:
df.pivot_table(index ='BMI Category', columns='Sleep Disorder', aggfunc={'Sleep Disorder':'count'}).plot.pie(autopct='%1.1f%%',
                                                                                                             subplots=True, figsize=(20,10))

plt.axis('equal')
plt.show()

## Data Preprocessing

In [ ]:
# check missing values
df.isnull().sum()

### for cat cols

In [ ]:
for col in cat_cols:
  print(f"{col}: {df[col].unique()}")

### Label Encoding of cat cols

In [ ]:
from sklearn.preprocessing import LabelEncoder


for col in cat_cols:
  le = LabelEncoder()

  #fit
  le.fit(df[col].unique())

  #encoder
  df[col] = le.transform(df[col])

  print(f"{col}: {df[col].unique()}")

## Correlation

In [ ]:
corr = df.corr()

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(corr, fmt='.2f', annot = True)
plt.title('Features affecting Sleep Disorder')
plt.show()

In [ ]:
max_5_corr = corr.nlargest(5, 'Sleep Disorder')
plt.figure(figsize=(15,5))
sns.heatmap(max_5_corr, fmt='.2f', annot = True)
plt.title('Max 5 Features affecting Sleep Disorder')
plt.show()


## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Sleep Disorder', axis =1 )
y = df['Sleep Disorder']

#Split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=42)

In [ ]:
print(f'Shape of X_train: {X_train.shape}')
print(f'Shape of y_train: {y_train.shape}')
print(f'Shape of X_test: {X_test.shape}')
print(f'Shape of y_test: {y_test.shape}')

# Model Training

##1. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()

In [ ]:
lr.fit(X_train, y_train)

In [ ]:
lr.score(X_train, y_train)

In [ ]:
lr.score(X_test, y_test)


In [ ]:
lr_pred = lr.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, jaccard_score, log_loss

accuracy_score(y_test, lr_pred)

In [ ]:
def model_accuracy(model, model_name):
    """ To print the accuracy score"""
    y_pred = model.predict(X_test)
    print(f"---------{model_name}---------")
    print(f'Accuracy : {round(accuracy_score(y_test, y_pred)*100,2)}%')
    print(f"F1 Score: {round(f1_score(y_test, y_pred, average = 'micro'), 2)}")
    print(f"Precision Score: {round(precision_score(y_test, y_pred, average = 'micro'), 2)}")
    print(f"Recall Score: {round(recall_score(y_test, y_pred, average='micro'), 2)}")
    print(f"Jaccard Score: {round(jaccard_score(y_test, y_pred, average='micro'), 2)}")

In [ ]:
model_accuracy(lr, 'Logistic Regression')

## 2. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dtree = DecisionTreeClassifier()
dtree.fit(X_train,y_train)
model_accuracy(dtree, "DecisionTree before GRID SEARCH")

In [ ]:
# paramgrid
param_grid = {
    'max_depth':[3,4,5,6,7,8],
    'min_samples_split':[2,3,4],
    'min_samples_leaf':[1,2,3,4],
    'random_state':[0,42]
}

In [ ]:
#cross validation
grid_search = GridSearchCV(dtree, param_grid, cv=5)
grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
dtree = DecisionTreeClassifier(
    random_state=42,
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=2
)

dtree.fit(X_train, y_train)
y_pred_tree = dtree.predict(X_test)
model_accuracy(dtree, "Decision Tree After GRID SEARCH")

In [ ]:
# top 10 important features
imp_df = pd.DataFrame({
    'Feature name' : X_train.columns,
    'importance': dtree.feature_importances_
}).sort_values(by='importance', ascending=False).head(10)
plt.figure(figsize=(10,8))
sns.barplot(data = imp_df, x = 'importance', y='Feature name')

plt.xlabel('Feature Importance')
plt.ylabel('Feature names')
plt.title('Top 10 features in terms of importance in Decision Tree')
plt.show()


In [ ]:
#confusion matrix

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_tree)
sns.heatmap(data = cm, linewidth=.5, annot=True, cmap="Blues")
plt.xlabel('Predicted values')
plt.ylabel('Actual Values')

## 2. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rforest = RandomForestClassifier()
rforest.fit(X_train,y_train)
model_accuracy(rforest, "RandomFOrest before CV")

In [ ]:
# GridSearch CV
param_grid={
    'n_estimators':[100,200],
    'max_depth':[None, 5, 10],
    'max_features':['sqrt','log2', None],
    'random_state':[0,42]
}

grid_searchrf = GridSearchCV(rforest, param_grid, cv=5)
grid_searchrf.fit(X_train, y_train)


In [ ]:
grid_searchrf.best_params_

In [ ]:
rforest2 = RandomForestClassifier(
    max_depth=5,
    max_features = 'sqrt',
    n_estimators = 100,
    random_state=0
)
rforest2.fit(X_train,y_train)
model_accuracy(rforest2, "RandomFOrest after CV")

In [ ]:
# top 10 important features
imp_df = pd.DataFrame({
    'Feature name' : X_train.columns,
    'importance': rforest2.feature_importances_
}).sort_values(by='importance', ascending=False).head(10)
plt.figure(figsize=(10,8))
sns.barplot(data = imp_df, x = 'importance', y='Feature name')

plt.xlabel('Feature Importance')
plt.ylabel('Feature names')
plt.title('Top 10 features in terms of importance in Random Forest calssifer')
plt.show()


In [ ]:
#confusion matrix

from sklearn.metrics import confusion_matrix
y_pred_rforest = rforest2.predict(X_test)
cm = confusion_matrix(y_test, y_pred_rforest)
sns.heatmap(data = cm, linewidth=.5, annot=True, cmap="Blues")
plt.xlabel('Predicted values')
plt.ylabel('Actual Values')
plt.title("Confusion Matrix for Random Forest")
plt.show()

In [ ]:
def makeConfusionMatrix(model, modelName):
  y_pred = model.predict(X_test)
  cm = confusion_matrix(y_test, y_pred)
  sns.heatmap(data = cm, linewidth=.5, annot=True, cmap="Blues")
  plt.xlabel('Predicted values')
  plt.ylabel('Actual Values')
  plt.title(f"Confusion Matrix for {modelName}")
  plt.show()

In [ ]:
makeConfusionMatrix(dtree, "decision Tree")

In [ ]:
makeConfusionMatrix(lr, 'Logisitic Regression')

In [ ]:
model_accuracy(dtree, 'Decision Tree')
model_accuracy(rforest2, "Random Forest")
model_accuracy(lr, "Logisitic Regression")

# Save the model


In [ ]:
import joblib
joblib.dump(dtree, 'DecisionTree.pkl')